In [0]:
USE CATALOG fraud_lakehouse;

In [0]:
SELECT current_user() AS me,
       is_account_group_member('admins') AS is_admin

In [0]:
SHOW GROUPS

In [0]:
SELECT is_member('admins') AS is_workspace_admin

In [0]:
GRANT USE CATALOG ON CATALOG fraud_lakehouse TO `fraud_analysts`;
GRANT USE SCHEMA ON SCHEMA gold TO `fraud_analysts`;
GRANT SELECT ON SCHEMA gold TO `fraud_analysts`;

GRANT USE CATALOG ON CATALOG fraud_lakehouse TO `risk_team`;
GRANT USE SCHEMA ON SCHEMA gold TO `risk_team`;
GRANT SELECT ON SCHEMA gold TO `risk_team`;

In [0]:
SHOW GRANTS ON SCHEMA gold

In [0]:
DESCRIBE gold.dim_customer

In [0]:
SELECT state, count(*) AS customers
FROM gold.dim_customer
WHERE is_current
GROUP BY state
ORDER BY customers DESC

In [0]:
CREATE OR REPLACE FUNCTION gold.state_filter(state STRING)
RETURN NOT is_account_group_member('fraud_analysts') OR state = 'Jalisco';

CREATE OR REPLACE FUNCTION gold.mask_card_last4(card_last4 STRING)
RETURN CASE WHEN is_account_group_member('risk_team') THEN card_last4 ELSE '****' END;

CREATE OR REPLACE FUNCTION gold.mask_credit_limit(credit_limit BIGINT)
RETURN CASE WHEN is_account_group_member('risk_team') THEN credit_limit END;

In [0]:
CREATE OR REPLACE VIEW gold.v_customer_secure AS
SELECT
  customer_sk,
  customer_id,
  gender,
  age_band,
  segment,
  gold.mask_credit_limit(credit_limit) AS credit_limit,
  city,
  state,
  gold.mask_card_last4(card_last4) AS card_last4,
  valid_from,
  valid_to,
  is_current
FROM gold.dim_customer
WHERE gold.state_filter(state)

In [0]:
SHOW TABLES IN gold

In [0]:
REVOKE SELECT ON SCHEMA gold FROM `fraud_analysts`;
REVOKE SELECT ON SCHEMA gold FROM `risk_team`;

GRANT SELECT ON TABLE gold.fact_transactions TO `fraud_analysts`;
GRANT SELECT ON TABLE gold.dim_merchant TO `fraud_analysts`;
GRANT SELECT ON TABLE gold.dim_date TO `fraud_analysts`;
GRANT SELECT ON TABLE gold.dim_category TO `fraud_analysts`;
GRANT SELECT ON TABLE gold.v_customer_secure TO `fraud_analysts`;

GRANT SELECT ON TABLE gold.fact_transactions TO `risk_team`;
GRANT SELECT ON TABLE gold.dim_merchant TO `risk_team`;
GRANT SELECT ON TABLE gold.dim_date TO `risk_team`;
GRANT SELECT ON TABLE gold.dim_category TO `risk_team`;
GRANT SELECT ON TABLE gold.v_customer_secure TO `risk_team`;

In [0]:
SHOW GRANTS ON TABLE gold.dim_customer

In [0]:
SELECT
  count(*) AS customers,
  count(DISTINCT state) AS states,
  count(credit_limit) AS visible_limits,
  max(card_last4) AS sample_card
FROM gold.v_customer_secure
WHERE is_current





In [0]:
SELECT
  is_account_group_member('fraud_analysts') AS in_fraud_analysts,
  is_account_group_member('risk_team') AS in_risk_team

In [0]:
SET TAG ON COLUMN gold.dim_customer.card_last4 pii = card;
SET TAG ON COLUMN gold.dim_customer.credit_limit pii = financial;
SET TAG ON COLUMN gold.dim_customer.home_lat pii = location;
SET TAG ON COLUMN gold.dim_customer.home_lon pii = location;

In [0]:
SELECT schema_name, table_name, column_name, tag_value
FROM information_schema.column_tags
WHERE tag_name = 'pii'
ORDER BY table_name, column_name